# Play dice with more than one agent

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

import altair as alt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

from dice.agents import Agent, SmartAgent, StopAtNDiceAgent
from dice.game import FillOrBustEnvironment, determine_available_actions

In [3]:
alt.renderers.set_embed_options(theme="dark")

RendererRegistry.enable('default')

# Utility functions

In [9]:
def plot_training_metric(
    agent: Agent,
    metric: str = "num_turns_to_win",
    downsample: bool = True,
) -> None:
    # Format the data
    data = getattr(agent, metric)
    if downsample:
        data = data[:: len(data) // min(2500, len(data))]
    source = (
        pd.DataFrame(data, columns=[metric])
        .reset_index()
        .rename(columns={"index": "episode"})
    )
    print(f"Median `{metric}` after training: {source.tail(50)[metric].median()}")

    # Create the chart
    padding = int(0.1 * len(source))
    line = (
        alt.Chart(source)
        .mark_line(color="red", size=3)
        .transform_window(
            rolling_mean=f"median({metric})",
            frame=[-padding, padding],
        )
        .encode(x="episode", y="rolling_mean:Q")
    )
    points = (
        alt.Chart(source)
        .mark_point(opacity=0.2, color="orange")
        .encode(x="episode", y=alt.Y(f"{metric}:Q").title(metric))
    )

    chart = (points + line).properties(width=400, height=250)
    chart.show()

# Train an agent

In [31]:
n_episodes = 100000
start_learning_rate = 0.01
lr_decay = start_learning_rate / (n_episodes // 2)
final_lr = 0.001
start_epsilon = 1.0
epsilon_decay = start_epsilon / (n_episodes // 1.5)
final_epsilon = 0.1
discount_factor = 0.98

kwargs = dict(
    initial_learning_rate=start_learning_rate,
    learning_rate_decay=lr_decay,
    final_learning_rate=final_lr,
    initial_epsilon=start_epsilon,
    epsilon_decay=epsilon_decay,
    final_epsilon=final_epsilon,
    discount_factor=discount_factor,
)

In [32]:
agent1 = SmartAgent("agent1", **kwargs)
agent2 = SmartAgent("agent2", **kwargs)

env = FillOrBustEnvironment()
for episode in tqdm(range(n_episodes)):
    obs, info = env.reset_for_new_game()
    done, turn_is_over = False, False

    # play one episode
    while not done:
        for agent in [agent1, agent2]:
            while not turn_is_over:
                action = agent.get_action(obs)
                next_obs, reward, terminated, truncated, turn_is_over, info = env.step(
                    action
                )

                # update the agent
                agent.update(obs, action, reward, terminated, next_obs, info)

                # update if the environment is done and the current obs
                done = terminated or truncated
                obs = next_obs
            turn_is_over = False

    for agent in [agent1, agent2]:
        agent.decay_epsilon()
        agent.decay_learning_rate()

100%|███████████████████████████████████████████████████████████████████████████████| 100000/100000 [03:58<00:00, 419.37it/s]


In [33]:
plot_training_metric(agent1, "num_turns_to_win")

Median `num_turns_to_win` after training: 44.0


alt.LayerChart(...)

In [34]:
plot_training_metric(agent2, "num_turns_to_win")

Median `num_turns_to_win` after training: 46.5


alt.LayerChart(...)

In [35]:
plot_training_metric(agent1, "rewards")

Median `rewards` after training: 0.0


alt.LayerChart(...)

In [36]:
plot_training_metric(agent2, "rewards")

Median `rewards` after training: 0.0


alt.LayerChart(...)